# Capítulo 2 — Calcular con vectores y matrices · Solución

*Matemáticas para la IA, paso a paso*, Belk Siri — notebook complementario.

[Abrir en Google Colab](https://colab.research.google.com/github/belksiri/books/blob/main/maths-pour-l-ia-pas-a-pas/notebooks/es/ch02_corrige.ipynb) · Página complementaria: https://belksiri.github.io/books/maths-pour-l-ia-pas-a-pas/es.html

Cada respuesta empieza con el razonamiento en papel y luego muestra su equivalente en Python.

**Regla de predicción del capítulo** (dada, no ajustada): pesos $\mathbf{w} = (3 ; -8)$ en k€/m² y k€/km, ordenada en el origen $b = 45$ k€.

Con los barrios (lección 2.4), la regla pasa a ser $\mathbf{w} = (3 ; -8 ; 12 ; -10)$, $b = 45$ k€, con Gare (la estación) como barrio de referencia.

**Índice.** Datos · Los cálculos de las lecciones · Ejercicios 2.1 a 2.6

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Colores del libro: un color = un papel (observado, predicho, desviación, recta del colega)
BLUE, ORANGE, RESIDUAL, GREEN = '#2a78d6', '#eb6834', '#c2185b', '#2e8b57'

## Datos

Los identificadores (D2.1, D2.E2) son los del libro y de los archivos CSV.

In [ ]:
# Datos del capítulo 2, copiados de los archivos CSV del libro (no modificar)

# D2.1 — las siete viviendas con su distancia al centro (lecciones 2.1 a 2.3)
homes = pd.DataFrame({
    'home':    ['A', 'B', 'C', 'D', 'E', 'F', 'G'],
    'floor_area':  [32, 45, 54, 70, 85, 98, 62],
    'distance': [3.0, 1.5, 2.5, 1.0, 2.5, 0.5, 2.5],
    'price':   [118, 172, 185, 245, 279, 335, 209],
})

# D2.4 — cinco viviendas del este de la ciudad, con su barrio (lección 2.4)
east = pd.DataFrame({
    'home':    ['K', 'L', 'M', 'N', 'O'],
    'district':    ['Parc', 'Rocade', 'Gare', 'Parc', 'Rocade'],
    'floor_area':  [60, 60, 60, 45, 80],
    'distance': [2.0, 2.0, 2.0, 1.0, 3.0],
    'price':   [223, 200, 210, 186, 248],
})

# D2.5 — los cinco T2 del Sur con su distancia al centro (lección 2.5)
t2_south = pd.DataFrame({
    'sale':       ['S03', 'S04', 'S05', 'S06', 'S07'],
    'floor_area':  [43, 40, 44, 46, 52],
    'distance': [3.5, 2.0, 1.5, 2.5, 0.5],
})

# D2.E2 — tres viviendas puestas a la venta (ejercicio 2.2)
new_homes = pd.DataFrame({
    'home':    ['H', 'I', 'J'],
    'floor_area':  [40, 75, 90],
    'distance': [0.5, 3.5, 1.0],
})

### Las variables de los datos

Los nombres del código están en inglés. Para las columnas, la última columna da el nombre utilizado en los archivos CSV del libro.

| Nombre | Significado | Unidad | Columna del CSV |
|:--|:--|:--|:--|
| `home` | identificador de la vivienda (A, B…) | — | `logement` |
| `floor_area` | superficie habitable de la vivienda | m² | `surface_m2` |
| `price` | precio de venta | k€ | `prix_kEUR` |
| `distance` | distancia al centro de la ciudad | km | `distance_km` |
| `district` | barrio de la vivienda | — | `quartier` |
| `sale` | identificador de la venta (S01…) | — | `vente` |

## Los cálculos de las lecciones, en Python

Estas celdas rehacen los cálculos de las lecciones 2.1 a 2.5. Ejecútalas y compáralas con el libro.

In [ ]:
# Lección 2.1 — la regla de la agencia aplicada a G, término a término (ecuación 2.1)
area_part = 3 * 62        # k€/m² × m² = k€
distance_part = -8 * 2.5     # k€/km × km = k€
fixed_part = 45               # k€
price_G = area_part + distance_part + fixed_part
print(area_part, distance_part, fixed_part, '→', price_G, 'k€; desviación con signo:', 209 - price_G, 'k€')

In [ ]:
# Lección 2.2 — vectores, producto escalar (ecuación 2.2) y predicción (ecuación 2.3)
x_G = np.array([62, 2.5])    # (superficie en m², distancia en km)
w = np.array([3, -8])        # (k€/m², k€/km), mismo orden que x
b = 45                       # k€
print('dimensión de x_G:', x_G.shape, '  w · x_G =', np.dot(w, x_G), '  ŷ =', np.dot(w, x_G) + b)
print('pesos en el orden equivocado:', np.dot(w[::-1], x_G) + b, 'k€ (absurdo)')

In [ ]:
# Lección 2.3 — todas las predicciones de una vez (ecuación 2.5)
X = homes[['floor_area', 'distance']].to_numpy(dtype=float)
y = homes['price'].to_numpy(dtype=float)
y_hat = X @ w + b
print('X :', X.shape, ' w :', w.shape, ' ŷ :', y_hat.shape)
print('ŷ =', y_hat)
print('desviaciones con signo y − ŷ =', y - y_hat)
print('media de las desviaciones en valor absoluto, 7 viviendas:', round(np.mean(np.abs(y - y_hat)), 2), 'k€')
print('ídem, A a F:', round(np.mean(np.abs(y - y_hat)[:6]), 2), 'k€; recta de la apertura del capítulo, A a F:',
      round(np.mean(np.abs(y - (3 * X[:, 0] + 30))[:6]), 2), 'k€')

In [ ]:
# Lección 2.4 — columnas indicadoras (ecuación 2.6), la Gare como referencia
dummies = pd.get_dummies(east['district'], drop_first=True).astype(int)   # columnas Parc, Rocade
X_east = np.column_stack([east['floor_area'], east['distance'], dummies['Parc'], dummies['Rocade']])
w_east = np.array([3, -8, 12, -10])
y_east = east['price'].to_numpy(dtype=float)
print(dummies.columns.tolist(), ' X:', X_east.shape)
print('con el barrio:', X_east @ w_east + 45)
print('sin el barrio:', X_east[:, :2] @ w_east[:2] + 45)
print('media de las desviaciones absolutas: sin', np.mean(np.abs(y_east - (X_east[:, :2] @ w_east[:2] + 45))),
      ' con', np.mean(np.abs(y_east - (X_east @ w_east + 45))))

In [ ]:
# Lección 2.5 — la misma predicción en dos escalas (ecuación 2.7), división por n
X5 = t2_south[['floor_area', 'distance']].to_numpy(dtype=float)
mu, sigma = X5.mean(axis=0), X5.std(axis=0, ddof=0)
w, b = np.array([3, -8]), 45
w_std, b_std = w * sigma, b + np.sum(w * mu)
Z = (X5 - mu) / sigma
print('medias:', mu, ' desviaciones típicas:', sigma)
print('pesos estandarizados:', w_std, ' ordenada:', b_std)
print('unidades originales:', X5 @ w + b)
print('escala estandarizada:', Z @ w_std + b_std)

## Ejercicio 2.1 — Leer un esquema

*Conceptos: producto de una matriz por un vector (lección 2.3).* Retoma el esquema matricial de la lección 2.3 (las siete viviendas A a G). La celda siguiente reconstruye $X$, $\mathbf{w}$ y $b$ para comprobar tus respuestas.

In [ ]:
X = homes[['floor_area', 'distance']].to_numpy(dtype=float)
w, b = np.array([3, -8]), 45
print(X)

In [ ]:
dimensions = {'X': (7, 2), 'w': (2,), 'y_hat': (7,)}
row_280 = 'E'
dimensions_c = {'X': (8, 3), 'w': (3,), 'y_hat': (8,)}

In [ ]:
# Comprobación
y_hat = X @ w + b
truth = {'X': X.shape, 'w': w.shape, 'y_hat': y_hat.shape}
if None in dimensions.values() or row_280 is None or None in dimensions_c.values():
    print('Completa la celda anterior.')
else:
    print('a) dimensiones:', 'correcto' if dimensions == truth else 'revisar')
    letter = homes['home'][int(np.argmin(np.abs(y_hat - 280)))]
    print('b) vivienda:', 'correcto' if row_280 == letter else 'revisar')
    X8 = np.zeros((8, 3)); w3 = np.zeros(3)   # solo la forma: 8 viviendas, 3 características
    print('c) dimensiones:', 'correcto' if dimensions_c == {'X': X8.shape, 'w': w3.shape,
          'y_hat': (X8 @ w3).shape} else 'revisar')

### Solución del ejercicio 2.1

**a)** $X$: **7 × 2** (7 viviendas, 2 características). $\mathbf{w}$: **2** componentes. $\hat{\mathbf{y}}$: **7** predicciones.

**b)** Es la fila **E**: 85 m² y 2,5 km. Con $\mathbf{w} = (3 ; -8)$ y $b = 45$: 3 × 85 + (−8) × 2,5 + 45 = 255 − 20 + 45 = **280 k€**.

**c)** Con la planta y una octava vivienda: $X$ pasa a ser **8 × 3**, $\mathbf{w}$ tiene **3** componentes (un peso para la planta), $\hat{\mathbf{y}}$ cuenta **8** predicciones. Si solo añades una vivienda, $\mathbf{w}$ conserva sus **2** componentes: el número de pesos depende de las características, no del número de viviendas. $X$ pasa entonces a ser 8 × 2 y $\hat{\mathbf{y}}$ cuenta 8 predicciones.

*Elementos esperados:* las tres dimensiones; los cinco números usados para E; la distinción entre filas (viviendas) y columnas (características).

## Ejercicio 2.2 — Tres viviendas nuevas

*Datos D2.E2. Conceptos: producto escalar (lección 2.2), producto de una matriz por un vector (lección 2.3).* Las preguntas a) a c) se hacen primero en papel, con la tabla del ejercicio 2.2 del libro.

In [ ]:
new_homes

### a) Escribir

$\mathbf{x}_H = (40 ; 0{,}5)$, $\mathbf{x}_I = (75 ; 3{,}5)$, $\mathbf{x}_J = (90 ; 1{,}0)$, con la superficie en m² y luego la distancia en km. La matriz reúne estos vectores como filas, en el mismo orden:

$$X = \begin{pmatrix} 40 & 0{,}5 \\ 75 & 3{,}5 \\ 90 & 1{,}0 \end{pmatrix}$$

Tiene 3 filas (una por vivienda) y 2 columnas (una por característica): sus dimensiones son **3 × 2**.

### b) Calcular a mano

| Vivienda | superficie: 3 k€/m² × … | distancia: −8 k€/km × … | parte fija | precio predicho |
|:--:|:--|:--|--:|--:|
| H | 3 × 40 = 120 k€ | −8 × 0,5 = −4 k€ | 45 k€ | **161 k€** |
| I | 3 × 75 = 225 k€ | −8 × 3,5 = −28 k€ | 45 k€ | **242 k€** |
| J | 3 × 90 = 270 k€ | −8 × 1,0 = −8 k€ | 45 k€ | **307 k€** |

Cada término está en k€: el m² se anula con el «por m²», el km con el «por km».

In [ ]:
price_by_hand = {'H': 161, 'I': 242, 'J': 307}

In [ ]:
# Comprobación (cálculo independiente, fila por fila)
for _, l in new_homes.iterrows():
    truth = 3 * l['floor_area'] - 8 * l['distance'] + 45
    yours = price_by_hand[l['home']]
    status = 'aún sin completar' if yours is None else ('correcto' if np.isclose(yours, truth) else 'revisar')
    print(f"{l['home']} : {status}")

### c) Interpretar

Recta de la apertura del capítulo (3 × superficie + 30): H → 150 k€, I → 255 k€, J → 300 k€. Diferencias (nueva regla − recta): H **+11**, I **−13**, J **+7** k€. La predicción cambia más para **I**, la vivienda más alejada del centro (3,5 km). Baja 13 k€.

*Para ir más lejos.* La diferencia entre las dos reglas vale $(3x_1 - 8x_2 + 45) - (3x_1 + 30) = 15 - 8x_2$: solo depende de la distancia. Es positiva por debajo de 15/8 ≈ 1,9 km y negativa por encima. H (0,5 km) gana 15 − 4 = 11 k€, I (3,5 km) pierde 28 − 15 = 13 k€, J (1 km) gana 15 − 8 = 7 k€.

*Elementos esperados:* los tres precios de la recta; las tres diferencias; I identificada y relacionada con su distancia.

### d) Todas las predicciones en una línea

Completa la celda y luego ejecuta la comprobación.

In [ ]:
X = new_homes[['floor_area', 'distance']].to_numpy(dtype=float)   # 3 × 2
w = np.array([3, -8])    # k€/m², k€/km
b = 45                   # k€
y_hat = X @ w + b    # ecuación 2.5

`X @ w` calcula los tres productos escalares (cada vez una fila de $X$ con $\mathbf{w}$). Después NumPy suma $b$ a cada uno.

In [ ]:
# Comprobación
if y_hat is None:
    print('Completa la celda anterior.')
else:
    print('dimensiones:', np.shape(X), '@', np.shape(w), '→', np.shape(y_hat))
    print('dimensiones:', 'correcto' if np.shape(X) == (3, 2) and np.shape(y_hat) == (3,) else 'revisar')
    print('predicciones:', y_hat, '→',
          'correcto' if np.allclose(y_hat, [161, 242, 307]) else 'revisar')

### e) Predecir y luego comprobar

**Predicción.** El peso −8 multiplica ahora la superficie (decenas de m²) y el peso 3 la distancia (unos pocos km). El término −8 × superficie domina con mucho: los tres precios serán **negativos**.

**En papel.** H: 3 × 0,5 − 8 × 40 + 45 = 1,5 − 320 + 45 = −273,5 k€. I: 3 × 3,5 − 8 × 75 + 45 = 10,5 − 600 + 45 = −544,5 k€. J: 3 × 1 − 8 × 90 + 45 = 3 − 720 + 45 = −672 k€.

**¿Por qué no hay error?** NumPy solo comprueba las dimensiones: 3 × 2 multiplicado por 2 da 3. Siempre son compatibles, y NumPy no conoce ni el significado ni la unidad de las columnas. Te toca a ti mantener el mismo orden en $X$ y en $\mathbf{w}$. Un control sencillo consiste en comprobar que un precio predicho es plausible, o en conservar los nombres de las columnas con pandas.

In [ ]:
if y_hat is not None:
    X_inverse = X[:, ::-1]          # columnas intercambiadas: distancia, superficie
    print('columnas intercambiadas:', X_inverse @ w + b)

## Ejercicio 2.3 — ¿Qué dicen los pesos?

*Conceptos: lectura de un peso (lección 2.2), columnas indicadoras (lección 2.4).* Regla: $\mathbf{w} = (3 ; -8 ; 12 ; -10)$ para superficie, distancia, Parc, Rocade; $b = 45$ k€; Gare es la referencia. Preguntas en papel, comprobación abajo.

In [ ]:
a_park_minus_ring_road = 12 - (-10)        # = 22 k€
b_far_minus_near = -8 * 1.5     # = −12 k€
c_far_park_minus_station = 12 - 8 * 1.5   # = 0 k€

In [ ]:
# Comprobación: calculamos la regla para pares de viviendas ficticias
rule = lambda floor_area, distance, park, ring_road: 3 * floor_area - 8 * distance + 12 * park - 10 * ring_road + 45
truths = {'a': rule(60, 2, 1, 0) - rule(60, 2, 0, 1),
         'b': rule(60, 3.5, 0, 0) - rule(60, 2, 0, 0),
         'c': rule(60, 3.5, 1, 0) - rule(60, 2, 0, 0)}
for k, yours in [('a', a_park_minus_ring_road), ('b', b_far_minus_near), ('c', c_far_park_minus_station)]:
    status = 'aún sin completar' if yours is None else ('correcto' if np.isclose(yours, truths[k]) else 'revisar')
    print(f'{k}) {status}')

### Solución del ejercicio 2.3

**a)** Solo difieren los términos de barrio: $12 \times 1 - (-10) \times 1 = 12 + 10 =$ **22 k€** más en Parc que en Rocade.

**b)** Solo difiere el término de distancia: $-8 \text{ k€/km} \times 1{,}5 \text{ km} =$ **−12 k€**. El piso más alejado se predice 12 k€ más barato.

**c)** Los dos efectos se compensan: +12 k€ por Parc, −12 k€ por los 1,5 km adicionales. La regla predice **el mismo precio**. Cada peso se lee «en igualdad de las demás condiciones». Cuando dos características cambian a la vez, se suman sus efectos.

**d)** $b = 45$ k€ es el precio predicho para una vivienda de 0 m², situada a 0 km del centro, en el barrio de referencia (Gare). Una vivienda así no existe: $b$ fija el nivel de la regla, como los 30 k€ de la apertura del capítulo.

*Elementos esperados:* las tres diferencias con su unidad; la idea de compensación; la lectura «respecto a Gare» de los pesos de barrio; $b$ sin realidad física.

## Ejercicio 2.4 — Columnas indicadoras en Python

*Datos D2.4. Conceptos: columnas indicadoras (lección 2.4), producto de una matriz por un vector (lección 2.3).*

In [ ]:
east

In [ ]:
park = (east['district'] == 'Parc').astype(int).to_numpy()
ring_road = (east['district'] == 'Rocade').astype(int).to_numpy()
X4 = np.column_stack([east['floor_area'], east['distance'], park, ring_road])
w4 = np.array([3, -8, 12, -10])
b4 = 45
y4 = X4 @ w4 + b4

**Variante.** `pd.get_dummies(east['district'], drop_first=True)` crea directamente las columnas Parc y Rocade, quitando la Gare, primera categoría por orden alfabético. Comprueba siempre qué categoría se ha quitado: es la que sirve de referencia para leer los pesos.

In [ ]:
# Comprobación
if y4 is None:
    print('Completa la celda anterior.')
else:
    print('columnas indicadoras:', 'correcto' if list(park) == [1, 0, 0, 1, 0] and list(ring_road) == [0, 1, 0, 0, 1] else 'revisar')
    print('X4 :', np.shape(X4), '→', 'correcto' if np.shape(X4) == (5, 4) else 'revisar')
    print('predicciones:', y4, '→', 'correcto' if np.allclose(y4, [221, 199, 209, 184, 251]) else 'revisar')

**c)** Media de las desviaciones en valor absoluto, con y sin las columnas de barrio.

In [ ]:
y_obs = east['price'].to_numpy(dtype=float)
mae_without = np.mean(np.abs(y_obs - (X4[:, :2] @ w4[:2] + b4)))
mae_with = np.mean(np.abs(y_obs - y4))

In [ ]:
if mae_without is None or mae_with is None:
    print('Completa la celda anterior.')
else:
    print(f'sin: {mae_without:.1f} k€, con: {mae_with:.1f} k€ →',
          'correcto' if np.isclose(mae_without, 10.2) and np.isclose(mae_with, 1.8) else 'revisar')

### Solución del ejercicio 2.4

**a)** Parc: (1, 0, 0, 1, 0). Rocade: (0, 1, 0, 0, 1). M, en Gare, tiene dos ceros.

**b)** $\hat{\mathbf{y}} = (221 ; 199 ; 209 ; 184 ; 251)$ k€. Por ejemplo, para N: 3 × 45 − 8 × 1 + 12 × 1 − 10 × 0 + 45 = 135 − 8 + 12 + 45 = 184 k€.

**c)** Sin el barrio, las desviaciones con signo valen +14, −9, +1, +14, −13: media de los valores absolutos 51 / 5 = **10,2 k€**. Con el barrio: +2, +1, +1, +2, −3, es decir 9 / 5 = **1,8 k€**. Como en la lección, estos números describen estas cinco ventas. No miden lo que daría la regla en ventas futuras.

## Ejercicio 2.5 — Predecir y luego comprobar: cambiar de unidad

*Datos D2.5. Conceptos: cambio de escala (lección 2.5, ecuación 2.7).* Anota tus predicciones para a) y b) antes de ejecutar las celdas.

**Predicciones esperadas.** a) El peso de la distancia pasa a ser −0,008 k€ por metro. La ordenada en el origen sigue siendo 45 k€, y las cinco predicciones no cambian. b) La media pasa de 2 km a 2000 m, la desviación típica de 1 km a 1000 m. El peso estandarizado sigue siendo $w'_2 = -0{,}008 \times 1\,000 = -8$ k€ por desviación típica: no depende de la unidad elegida.

In [ ]:
def standardized_weights(w, b, mu, sigma):
    w_std = w * sigma              # k€ por desviación típica
    b_std = b + np.sum(w * mu)     # precio predicho de la vivienda media
    return w_std, b_std

In [ ]:
# Dos versiones de la misma regla: distancia en km y luego en metros
X_km = t2_south[['floor_area', 'distance']].to_numpy(dtype=float)
X_m = X_km * np.array([1, 1000])
w_km, w_m, b = np.array([3, -8]), np.array([3, -0.008]), 45
print('predicciones (km):', X_km @ w_km + b)
print('predicciones (m): ', X_m @ w_m + b)
results = {}
for name, Xv, wv in [('km', X_km, w_km), ('m', X_m, w_m)]:
    mu, sigma = Xv.mean(axis=0), Xv.std(axis=0, ddof=0)
    results[name] = standardized_weights(wv, b, mu, sigma)
    print(name, ': media distancia =', mu[1], ', desviación típica =', sigma[1], ', pesos estandarizados =', results[name])

In [ ]:
# Comprobación
if results['km'][0] is None:
    print('Completa la función standardized_weights.')
else:
    ok = all(np.allclose(results[k][0], [12, -8]) and np.isclose(results[k][1], 164) for k in results)
    print('pesos estandarizados idénticos en km y en m:', 'correcto' if ok else 'revisar')

In [ ]:
if results['km'][0] is not None:
    fig, axes = plt.subplots(1, 2, figsize=(8, 2.6))
    for ax, title, values in [
            (axes[0], 'Peso bruto del término 2', {'km': w_km[1], 'm': w_m[1]}),
            (axes[1], 'Peso estandarizado del término 2', {k: results[k][0][1] for k in results})]:
        ax.bar(list(values), list(values.values()), color=[BLUE, ORANGE])
        ax.axhline(0, color='black', linewidth=0.8)
        ax.set_title(title, fontsize=10)
    plt.tight_layout()
    plt.show()

### Solución del ejercicio 2.5

**a)** Un metro vale 1/1000 km: el peso pasa a ser −8 / 1000 = **−0,008 k€/m**. Como cada distancia se multiplica por 1000, cada producto $w_2 x_2$ sigue siendo idéntico: la ordenada en el origen (45 k€) y las cinco predicciones (146, 149, 165, 163, 197 k€) **no cambian**.

**b)** Media 2000 m, desviación típica 1000 m. Con la ecuación 2.7: $w'_2 = -0{,}008 \times 1\,000 = -8$ y $b' = 45 + 3 \times 45 - 0{,}008 \times 2\,000 = 45 + 135 - 16 = 164$ k€, **exactamente** como en km.

**c)** En el gráfico de la izquierda, el peso bruto pasa de −8 a −0,008: su tamaño depende de la unidad y no permite ninguna comparación. En el de la derecha, los pesos estandarizados coinciden: un aumento de una desviación típica de la distancia corresponde siempre a −8 k€ en la predicción, con la superficie fija. Esta lectura describe la regla sobre estos datos. No mide ni una importancia general ni un efecto causal.

*Elementos esperados:* predicciones sin cambios; peso bruto dividido por 1000; peso estandarizado sin cambios; conclusión sobre la comparación de los pesos.

## Ejercicio 2.6 — Reto: una regla, varias escrituras

*Datos D2.4. Conceptos: lecciones 2.3, 2.4 y 2.5.* Tres columnas indicadoras: Parc, Rocade, Gare.

In [ ]:
station = (east['district'] == 'Gare').astype(int).to_numpy()
park3 = (east['district'] == 'Parc').astype(int).to_numpy()
ring_road3 = (east['district'] == 'Rocade').astype(int).to_numpy()
X3 = np.column_stack([east['floor_area'], east['distance'], park3, ring_road3, station])
print('suma de las tres indicadoras, fila por fila:', X3[:, 2:].sum(axis=1))
print('regla del colega (b = 40):', X3 @ np.array([3, -8, 17, -5, 5]) + 40)

In [ ]:
district_weight_b50 = [7, -15, -5]   # cada peso de barrio disminuye 5 cuando b aumenta 5

In [ ]:
# Comprobación
if district_weight_b50 is None:
    print('Completa la celda anterior.')
else:
    y50 = X3 @ np.array([3, -8] + list(district_weight_b50)) + 50
    print('predicciones con b = 50:', y50, '→', 'correcto' if np.allclose(y50, [221, 199, 209, 184, 251]) else 'revisar')

### Solución del ejercicio 2.6

**a)** K: 180 − 16 + 17 + 40 = 221; L: 180 − 16 − 5 + 40 = 199; M: 180 − 16 + 5 + 40 = 209; N: 135 − 8 + 17 + 40 = 184; O: 240 − 24 − 5 + 40 = 251. Son **exactamente** las predicciones de la tabla de la lección 2.4.

**b)** Cada vivienda tiene un único 1 entre las tres indicadoras. Sumar una misma cantidad $c$ a los tres pesos de barrio suma, por tanto, $c$ a todas las predicciones, lo que se compensa restando $c$ a $b$. Para $b = 50$, $c = -5$ respecto a la regla de la lección (12, −10, 0, $b = 45$): pesos **(7 ; −15 ; −5)**. Cualquier $c$ sirve: existen infinitas reglas equivalentes.

**c)** En la ecuación 2.7 se cambian las *columnas* (unidades originales o puntuaciones estandarizadas), y a cada escritura de las columnas le corresponde una única regla. Es una traducción, sin ambigüedad. Con tres indicadoras, las columnas siguen siendo las mismas y, sin embargo, infinitos pesos dan las mismas predicciones. Es esta segunda situación la que estorba al ajuste del capítulo 3: nada en los datos permite elegir entre estas reglas. Quitar una indicadora, la de la referencia, elimina la ambigüedad.

*Elementos esperados:* las cinco predicciones; el argumento de que la suma de las indicadoras es 1; la distinción entre cambio de escala (regla única para unas columnas dadas) y redundancia (regla no única).